In [11]:
import pandas as pd

# Load without headers
df = pd.read_csv("/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Physiological_signals/cleaned_user1_physiological.csv",skiprows=1, header=None)

# Assign proper column names
df.columns = ['ArduinoTime_ms', 'GSR', 'Pulse', 'SystemTime_ms', 'Datetime']

# Preview
df.head(10)


,ArduinoTime_ms,GSR,Pulse,SystemTime_ms,Datetime
0,0,628,515,1751709909125,2025-07-05 10:05:09.125
1,86081228,628,516,1751709909225,2025-07-05 10:05:09.225
2,8600,628,515,1751709909325,2025-07-05 10:05:09.325
3,86081228,628,516,1751709909425,2025-07-05 10:05:09.425
4,8600,628,515,1751709909525,2025-07-05 10:05:09.525
5,86081228,628,516,1751709909625,2025-07-05 10:05:09.625
6,8600,628,515,1751709909725,2025-07-05 10:05:09.725
7,86081228,628,516,1751709909825,2025-07-05 10:05:09.825
8,8600,628,515,1751709909925,2025-07-05 10:05:09.925
9,86081228,628,516,1751709910025,2025-07-05 10:05:10.025


In [12]:
df.isna().sum()


ArduinoTime_ms    0
GSR               0
Pulse             0
SystemTime_ms     0
Datetime          0
dtype: int64

In [13]:
# Check for NaNs
df.isna().sum()

# View datatypes
df.dtypes


ArduinoTime_ms     int64
GSR                int64
Pulse              int64
SystemTime_ms      int64
Datetime          object
dtype: object

In [14]:
df.describe()


,ArduinoTime_ms,GSR,Pulse,SystemTime_ms
count,1.102170e+05,110217.000000,110217.000000,1.102170e+05
mean,1.479179e+05,627.858515,516.786503,1.751715e+12
std,1.921372e+06,1.889086,3.163843,3.181705e+06
min,0.000000e+00,618.000000,503.000000,1.751710e+12
25%,8.124800e+04,628.000000,518.000000,1.751713e+12
50%,8.124800e+04,628.000000,518.000000,1.751715e+12
75%,8.124800e+04,628.000000,518.000000,1.751718e+12
max,8.608125e+07,642.000000,529.000000,1.751721e+12


In [15]:
df.head()

,ArduinoTime_ms,GSR,Pulse,SystemTime_ms,Datetime
0,0,628,515,1751709909125,2025-07-05 10:05:09.125
1,86081228,628,516,1751709909225,2025-07-05 10:05:09.225
2,8600,628,515,1751709909325,2025-07-05 10:05:09.325
3,86081228,628,516,1751709909425,2025-07-05 10:05:09.425
4,8600,628,515,1751709909525,2025-07-05 10:05:09.525


In [16]:
import pandas as pd
import numpy as np

# # Load raw data
# df = pd.read_csv("raw_data/Physiological_signals/user1_physiological.csv", header=None)
# df.columns = ['ArduinoTime_ms', 'GSR', 'Pulse', 'SystemTime_ms', 'Datetime']

# Drop fully empty rows (just in case)
df = df.dropna(how='all')

# Step 1: Fix SystemTime_ms using sampling rate
# ---------------------------------------------
sampling_rate_hz = 10
interval_ms = 100  # 1 sample every 100ms

# Get first valid SystemTime
first_valid_ts = df['SystemTime_ms'].dropna().astype(np.float64).iloc[0]
first_valid_ts = int(first_valid_ts)

# Generate synthetic SystemTime_ms
df['SystemTime_ms_fixed'] = first_valid_ts + np.arange(len(df)) * interval_ms

# Step 2: Convert SystemTime_ms_fixed to Datetime
# ------------------------------------------------
df['Datetime_fixed'] = pd.to_datetime(df['SystemTime_ms_fixed'], unit='ms')

# Step 3: Drop or keep original time columns as needed
# -----------------------------------------------------
# You can keep them for reference, or just replace them:
df_cleaned = df.copy()
df_cleaned['SystemTime_ms'] = df_cleaned['SystemTime_ms_fixed']
df_cleaned['Datetime'] = df_cleaned['Datetime_fixed']

df_cleaned = df_cleaned.drop(columns=['SystemTime_ms_fixed', 'Datetime_fixed'])

# Step 4: Export cleaned data (optional)
# ---------------------------------------
df_cleaned.to_csv("cleaned_user1_physiological.csv", index=False)
print("[OK] Cleaned data saved as 'cleaned_user1_physiological.csv'")

# Optional: Check the time gaps to verify constant sampling
df_cleaned['delta'] = df_cleaned['SystemTime_ms'].diff()
print(df_cleaned['delta'].value_counts().head())


[OK] Cleaned data saved as 'cleaned_user1_physiological.csv'
delta
100.0    110216
Name: count, dtype: int64


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# File paths
ps_path = "/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Physiological_signals/cleaned_user1_physiological.csv"
annotation_path = "/home/rounak/CODE/Low_Engagement_Detection/Data_preprocess/Preprocess_Signals/raw_data/Annotations/user1_annotations.csv"

# Load data
PS = pd.read_csv(ps_path, skiprows=1)
PS.columns = ['time', 'GSR', 'HR', 'timestamp', 'time2']
annotation = pd.read_csv(annotation_path)

# Print basic info
print("Annotation video IDs:", annotation["videoID"].unique())
print("PS timestamp range:", PS["timestamp"].min(), "to", PS["timestamp"].max())

# Check rows with video 1 and 2
v1_index = annotation[annotation["videoID"] == 5].index.max()
v2_index = annotation[annotation["videoID"] == 3].index.min()

print("v1_index:", v1_index)
print("v2_index:", v2_index)

if pd.notna(v1_index) and pd.notna(v2_index):
    start_time = annotation.loc[v1_index, 'time']
    end_time = annotation.loc[v2_index, 'time']
    print("Start Time:", start_time)
    print("End Time:", end_time)

    # Extract baseline data
    baseline_df = PS[(PS['timestamp'] >= start_time) & (PS['timestamp'] <= end_time)]
    print("Baseline data points found:", len(baseline_df))

    # Plot if needed
    if not baseline_df.empty:
        plt.figure(figsize=(12, 5))
        plt.plot(baseline_df["timestamp"], baseline_df["GSR"], label="GSR")
        plt.plot(baseline_df["timestamp"], baseline_df["HR"], label="HR")
        plt.title("Baseline Signal (Video 1 to Video 2 Gap)")
        plt.xlabel("Timestamp")
        plt.ylabel("Normalized Signal")
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("⚠️ No data in baseline window — timestamps may not overlap.")
else:
    print("❌ Could not find video 1 or video 2 in annotation.")


Annotation video IDs: [0 5 3 7 8 1 2 4 6]
PS timestamp range: 1751709909225 to 1751720930725
v1_index: 1070
v2_index: 129
Start Time: 1751712291693
End Time: 1751710311929
Baseline data points found: 0
⚠️ No data in baseline window — timestamps may not overlap.
